<a href="https://colab.research.google.com/github/M-Anees-Jafar1/Email_and_SMS_Spam_Classifier_using_pytorch_GRU-/blob/main/Untitled18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
path='/content/drive/MyDrive/email.csv'
df=pd.read_csv(path)
print(df.head())
print(df.shape)


  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
(5573, 2)


In [15]:
df['Label']=df['Category'].map({'ham': 0 , 'spam': 1})
print(df[['Category','Label','Message']].head())

  Category  Label                                            Message
0      ham    0.0  Go until jurong point, crazy.. Available only ...
1      ham    0.0                      Ok lar... Joking wif u oni...
2     spam    1.0  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham    0.0  U dun say so early hor... U c already then say...
4      ham    0.0  Nah I don't think he goes to usf, he lives aro...


In [16]:
df=df.dropna(subset=['Label'])

In [17]:
df['Label']=df['Label'].astype(int)

In [18]:
import pandas as pd
import numpy as np
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

# ========================================================
# STEP 2: Automatic Vocab Generation
# ========================================================
all_words = []
for message in df['Message']:
    words = str(message).lower().split()
    all_words.extend(words)

word_counts = Counter(all_words)

# Vocab register initialize kiya
vocab = {'<PAD>': 0, '<UNK>': 1}
current_id = 2
for word, count in word_counts.items():
    if count >= 2:
        vocab[word] = current_id
        current_id += 1

print("--- 1. Vocabulary Built Successfully ---")
print("Total Unique Words in Vocab:", len(vocab))


# ========================================================
# STEP 4: Full Dataset Encoding & Padding
# ========================================================
max_len = 50
all_encoded_emails = []

for message in df['Message']:
    row_words = str(message).lower().split()

    encoded_seq = []
    for word in row_words:
        word_id = vocab.get(word, 1)
        encoded_seq.append(word_id)

    if len(encoded_seq) < max_len:
        encoded_seq = encoded_seq + [0] * (max_len - len(encoded_seq))
    else:
        encoded_seq = encoded_seq[:max_len]

    all_encoded_emails.append(encoded_seq)

X = torch.tensor(all_encoded_emails, dtype=torch.long)
y = torch.tensor(df['Label'].values, dtype=torch.long)

print("--- 2. Text to Tensors Complete ---")


# ========================================================
# STEP 5: Train-Test Split
# ========================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Data को Batches में डालना (Batch Size = 32)
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

print("--- 3. Data Split and Loader Ready ---")


# ========================================================
# STEP 6 & 7: Model Architecture, Loss & Optimizer
# ========================================================
class EmailGRUModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(EmailGRUModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        output, hidden = self.gru(embedded)
        return self.fc(hidden[0])

VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
OUTPUT_DIM = 2

model = EmailGRUModel(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)


# ========================================================
# STEP 8: Advanced Training Loop (15 Epochs)
# ========================================================
epochs = 15
print("\n--- 4. Advanced Training Started ---")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

    epoch_loss = total_loss / len(train_loader)
    epoch_acc = (correct / total) * 100
    print(f"Epoch [{epoch+1}/{epochs}] -> Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

print("--- Training Finished! ---")


# ========================================================
# STEP 9: Model Evaluation (Testing)
# ========================================================
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    test_loss = criterion(test_outputs, y_test)
    _, test_predicted = torch.max(test_outputs, 1)
    test_correct = (test_predicted == y_test).sum().item()
    test_accuracy = (test_correct / y_test.size(0)) * 100

print("\n--- 5. Testing Results ---")
print(f"Test Loss: {test_loss.item():.4f} | Test Accuracy: {test_accuracy:.2f}%\n")


# ========================================================
# STEP 10: Custom Inference Function
# ========================================================
def predict_spam_or_ham(custom_message):
    model.eval()
    words = str(custom_message).lower().split()
    encoded_seq = [vocab.get(word, 1) for word in words]

    if len(encoded_seq) < max_len:
        encoded_seq = encoded_seq + [0] * (max_len - len(encoded_seq))
    else:
        encoded_seq = encoded_seq[:max_len]

    input_tensor = torch.tensor([encoded_seq], dtype=torch.long)

    with torch.no_grad():
        output = model(input_tensor)
        _, predicted = torch.max(output, 1)

    result = "SPAM 🚨" if predicted.item() == 1 else "HAM (Normal) ✅"
    print(f"Message: '{custom_message}'")
    print(f"Prediction: {result}\n")

# --- Final Tests ---
print("--- 6. Custom Testing ---")
predict_spam_or_ham("Hey friend, are we still meeting for lunch today?")
predict_spam_or_ham("WINNER! You have won a free cash prize of 1000 dollars. Call now to claim!")

--- 1. Vocabulary Built Successfully ---
Total Unique Words in Vocab: 5531
--- 2. Text to Tensors Complete ---
--- 3. Data Split and Loader Ready ---

--- 4. Advanced Training Started ---
Epoch [1/15] -> Loss: 0.3547 | Accuracy: 87.17%
Epoch [2/15] -> Loss: 0.0846 | Accuracy: 97.20%
Epoch [3/15] -> Loss: 0.0432 | Accuracy: 98.81%
Epoch [4/15] -> Loss: 0.0182 | Accuracy: 99.62%
Epoch [5/15] -> Loss: 0.0128 | Accuracy: 99.57%
Epoch [6/15] -> Loss: 0.0069 | Accuracy: 99.87%
Epoch [7/15] -> Loss: 0.0040 | Accuracy: 99.87%
Epoch [8/15] -> Loss: 0.0024 | Accuracy: 99.89%
Epoch [9/15] -> Loss: 0.0016 | Accuracy: 99.96%
Epoch [10/15] -> Loss: 0.0024 | Accuracy: 99.96%
Epoch [11/15] -> Loss: 0.0034 | Accuracy: 99.89%
Epoch [12/15] -> Loss: 0.0014 | Accuracy: 99.93%
Epoch [13/15] -> Loss: 0.0013 | Accuracy: 99.96%
Epoch [14/15] -> Loss: 0.0012 | Accuracy: 99.96%
Epoch [15/15] -> Loss: 0.0014 | Accuracy: 99.96%
--- Training Finished! ---

--- 5. Testing Results ---
Test Loss: 0.1240 | Test Accura